# 2.2 Feature selection

**What this notebook is for.** Going from "every column in the matrix" to a
ranked, trimmed feature list, and writing that list somewhere the training sweep
can find it.

**Why selection happens at all.** The feature library emits hundreds of columns.
Most carry nothing: a count that is null for 99% of the population, a variance of
zero, a column that is really an identifier. Feeding those to a distance-based
detector is worse than useless, because every one of them adds a dimension to the
space the distance is measured in.

**The four stages.**

1. `classify_columns` labels every column: identifier, timestamp, provenance,
   label-leaking, or usable numeric. It returns `ColumnClassification` objects
   carrying a reason from `EXCLUSION_REASONS`, so an exclusion is always
   explainable.
2. `profile_features` computes null rate, variance and quantiles. It samples, and
   it uses approximate quantiles with a configurable `relative_error` — exact
   quantiles on a wide matrix are a full sort per column.
3. `rank_features` scores what survived. The score is variance-based with a boost
   for behavioural tokens, which is a *prior*, not a discovery: features about
   what a customer did are believed more useful than features about what they
   are. `FAMILY_BOOST_WEIGHT` is the knob.
4. `select_top_n` cuts to a length.

**The label-leakage list is not optional.** `LABEL_LEAKAGE_COLUMNS` names columns
derived from the fraud label itself. Leaving one in produces a model that scores
beautifully in evaluation and is worthless in production, and the failure is
silent because the metrics look *better*, not worse.

In [ ]:
import os
import sys
import warnings

warnings.filterwarnings("ignore")

# The notebooks live two levels below the repo root. Put the root on sys.path so
# `import trust_score_05` resolves without an editable install, which is what
# happens on EMR and SageMaker where the package arrives as a zip.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print("repo root:", REPO_ROOT)

In [ ]:
from trust_score_05.ml.config import load_ml_config

ENV = "dev"
cfg = load_ml_config(env=ENV)

MODEL = "isolation_forest"  # the ranking prior is per-model
print("model      :", MODEL)
print("top-n grid :", cfg.top_n_features)
print("join key   :", cfg.join_key)

In [ ]:
from trust_score_05.ml.jobs.base import build_spark_session

spark = build_spark_session("notebook")
spark.sparkContext.setLogLevel("WARN")
spark

## The matrix

In [ ]:
from trust_score_05.ml.datasets import load_training_matrix

training = load_training_matrix(spark, cfg, features=None)
training.cache()

dtypes = dict(training.dtypes)
print("rows    :", training.count())
print("columns :", len(dtypes))

## Stage 1 — classify

Read the exclusion reasons. Every column that is *not* a candidate should be
excluded for a reason you agree with; a feature you expected to see in the
candidate list and do not is the interesting case.

In [ ]:
import collections

from trust_score_05.ml.selection import (
    EXCLUSION_REASONS,
    LABEL_LEAKAGE_COLUMNS,
    candidate_features,
    classify_columns,
)

classified = classify_columns(dtypes, join_key=cfg.join_key)
candidates = candidate_features(dtypes, join_key=cfg.join_key)

reasons = collections.Counter(
    c.reason for c in classified.values() if not c.is_candidate
)
print(f"{len(candidates)} candidate(s) out of {len(dtypes)} column(s)")
print()
for reason, count in reasons.most_common():
    print(f"{count:4d}  {reason}")

print()
leaks = [c for c in classified.values() if c.reason == "label_leakage"]
print(f"label-leakage exclusions ({len(leaks)}):", [c.name for c in leaks])
print("the configured leakage list:", sorted(LABEL_LEAKAGE_COLUMNS))

## Stage 2 — profile

`sample_rows` caps the scan. `relative_error` trades quantile accuracy for time;
the default is fine for ranking and too coarse for reporting a percentile to a
stakeholder.

In [ ]:
from trust_score_05.ml.selection import profile_features

profile = profile_features(
    training,
    candidates,
    sample_rows=cfg.profile_sample_rows,
    seed=cfg.seed,
)

print("profiled:", len(profile))
profile.sort_values("null_rate", ascending=False).head(15)

## Stage 3 — rank

`max_null_rate` and `min_variance` are the hard cuts; everything that survives
gets a score. Look at the top and the bottom of the ranking, and look at *why*
the top ones are there — if the boost is doing all the work then the boost is the
model, which is a thing to know.

In [ ]:
from trust_score_05.ml.selection import RANKED_FEATURE_COLUMNS, rank_features

ranked = rank_features(profile, model=MODEL)

print("ranked columns:", list(ranked.columns))
print("survivors     :", len(ranked))
print()
print("top 25:")
ranked.head(25)[[c for c in RANKED_FEATURE_COLUMNS if c in ranked.columns]]

In [ ]:
from trust_score_05.ml.selection import FAMILY_BOOST_TOKENS, FAMILY_BOOST_WEIGHT

print("boost weight:", FAMILY_BOOST_WEIGHT)
print("boost tokens:", sorted(FAMILY_BOOST_TOKENS))
print()
boosted = ranked[ranked["family_boost"] > 0] if "family_boost" in ranked else ranked.iloc[:0]
print(f"{len(boosted)} of {len(ranked)} ranked features got a boost")

## Stage 4 — cut

In [ ]:
from trust_score_05.ml.selection import select_top_n

selected = select_top_n(ranked, counts=cfg.top_n_features)

for count, names in sorted(selected.items()):
    print(f"top {count:4d}: {len(names)} feature(s)")

largest = max(selected)
print()
print(f"the top-{largest} list:")
for name in selected[largest][:40]:
    print(" ", name)
if len(selected[largest]) > 40:
    print(f"  ... and {len(selected[largest]) - 40} more")

## Publish

`write_feature_selection` writes the ranked table, the selected lists and a
manifest under `feature_selection_prefix(cfg, model, method, run_id)`, then drops
a `_READY` marker. The sweep reads it back with `read_feature_selection`, and
resolves which run to read with `resolve_selection_run_id` — so a sweep is always
pinned to one selection run, and rerunning selection does not silently change
what a sweep in flight is training on.

In [ ]:
from trust_score_05.ml.jobs.base import write_feature_selection
from trust_score_05.ml.paths import feature_selection_prefix
from trust_score_05.ml.selection import METHOD_NAME

prefix = feature_selection_prefix(cfg, model=MODEL, method=METHOD_NAME)
print("would write to:", prefix)

PUBLISH = False

if PUBLISH:
    written = write_feature_selection(prefix, ranked=ranked, selected=selected)
    print("wrote:", written)
else:
    print("PUBLISH is False -- nothing written.")

## What next

`3.0-modeling/3.1-preprocess-and-fit.ipynb` fits one model on one feature list so
you can see it work. `3.2-sweep.ipynb` then does it across the whole grid.

The headless equivalent of this notebook is
`python -m trust_score_05.ml.jobs.selection_job --env dev --model isolation_forest`.